# P14: feed-forward networks on good features

P02 and P13 fed the feed-forward network raw pixels: it stays near 40% whatever the cure (ADR 0016). Here the same kind of
network gets **better inputs**, which separates two questions the director's "is a neural network worth it?" mixes up: is it
the *network* that pays, or the *features* it is given?

1. **On the handcrafted features of the classical baseline** (P11, ADR 0014: colour histograms and HOG edges, 1,802 numbers
   per painting). The random forest of P11 scored 60.0% on validation and 58.5% on museum B. With the *same* numbers as input,
   does a feed-forward network beat the forest?
2. **On the features of the frozen ResNet-18** (P06: 512 numbers per painting). Its head is a single linear layer (80.3%,
   5 seeds). Does a hidden layer on top, a small feed-forward head, do better?

**The rules, written before any run of this notebook** (a first scratch run, unsaved, checked the code and the run times):

- Every network: 5 seeds (0 to 4), Adam at 1e-3, batches of 64, best epoch kept on validation, saved with `run_experiment`.
- **Part 1:** eight networks fixed in advance (no hidden layer, i.e. a logistic regression trained by gradient descent; a hidden
  layer of 256 or 512 ReLUs; dropout 0 or 0.5; weight decay 0 or 1e-2), 50 epochs. The best 5-seed validation mean is chosen
  and scored **once on museum B**, whatever its validation score: it is a new kind of model for the director's comparison.
  The forest is re-run with 5 random states on validation (not on museum B) to know its own spread.
- **Part 2:** the linear head is re-run on this computer (5 seeds) and compared with two feed-forward heads (256 hidden ReLUs,
  dropout 0 or 0.5), 30 epochs. A head counts only if its mean beats the re-run linear head by more than 2 points (ADR 0011);
  only then is it scored on museum B.

In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestClassifier

from deep_learning_project_waerniers_lamy_mahieddine.vision.utils import (
    DATA_DIR, build_feature_mlp, build_resnet_head, load_handcrafted, load_resnet_data, run_experiment, score_on_test, summary)

SEEDS = 5
BAR = 0.02
runs = {}


def record(label, name, config, build_model, load_fn):
    """run_experiment with 5 seeds, kept under a readable label."""
    print(f"===== {label}")
    runs[label] = run_experiment(name, config, seeds=SEEDS, build_model=build_model, load_fn=load_fn)
    accs = runs[label]["val_acc_by_seed"]
    print(f"===== {label}: mean {np.mean(accs):.1%} (std {np.std(accs, ddof=1):.1%}, {min(accs):.1%} to {max(accs):.1%})\n")


def table(labels):
    return pd.DataFrame([{"network": l, "val mean (5 seeds)": np.mean(runs[l]["val_acc_by_seed"]),
                          "std": np.std(runs[l]["val_acc_by_seed"], ddof=1), "min": min(runs[l]["val_acc_by_seed"]),
                          "max": max(runs[l]["val_acc_by_seed"]),
                          "trained weights": sum(p.numel() for p in runs[l]["model"].parameters())}
                         for l in labels]).set_index("network").round(3)

## 1. A feed-forward network on the handcrafted features

The 1,802 numbers of P11, computed by the same function (now `handcrafted_features` in `utils.py`), standardised with the
mean and std of the training paintings (a forest does not need it, a network does: P12, section 3).

In [2]:
start = time.time()
features = load_handcrafted()
print(f"train {tuple(features['X_train'].shape)}, validation {tuple(features['X_val'].shape)} ({time.time() - start:.0f} s, cached after the first run)")

# The P11 forest, same settings, with 5 random states: its own spread on validation (museum B is not used here)
raw = np.load(DATA_DIR / "handcrafted.npz")  # the unstandardised features, as the forest of P11 used them
split = pd.read_csv(DATA_DIR / "split.csv")["split"].to_numpy()
forest_accs = []
for state in range(SEEDS):
    forest = RandomForestClassifier(300, n_jobs=-1, random_state=state).fit(raw["X"][split == "train"], features["y_train"].numpy())
    forest_accs.append((forest.predict(raw["X"][split == "val"]) == features["y_val"].numpy()).mean())
print("random forest, 300 trees, random states 0-4:", [f"{a:.1%}" for a in forest_accs],
      f"| mean {np.mean(forest_accs):.1%} (std {np.std(forest_accs, ddof=1):.1%}) | P11 recorded 60.0% for state 0")

train (1920, 1802), validation (480, 1802) (0 s, cached after the first run)


random forest, 300 trees, random states 0-4: ['60.4%', '59.4%', '57.7%', '63.7%', '62.3%'] | mean 60.7% (std 2.4%) | P11 recorded 60.0% for state 0


In [3]:
FEATURE_BASE = {"model": "mlp_features", "image_size": 128, "n_features": features["X_train"].shape[1], "epochs": 50,
                "lr": 1e-3, "batch_size": 64, "weight_decay": 0.0, "dropout": 0.0, "seed": 0}
GRID = {  # label: (saved name, changes), fixed before the runs
    "linear": ("features_linear", {"hidden": 0}),
    "linear, wd 1e-2": ("features_linear_wd", {"hidden": 0, "weight_decay": 1e-2}),
    "256": ("features_mlp256", {"hidden": 256}),
    "256, wd 1e-2": ("features_mlp256_wd", {"hidden": 256, "weight_decay": 1e-2}),
    "256, dropout 0.5": ("features_mlp256_drop", {"hidden": 256, "dropout": 0.5}),
    "256, dropout 0.5, wd 1e-2": ("features_mlp256_drop_wd", {"hidden": 256, "dropout": 0.5, "weight_decay": 1e-2}),
    "512, dropout 0.5": ("features_mlp512_drop", {"hidden": 512, "dropout": 0.5}),
    "512, dropout 0.5, wd 1e-2": ("features_mlp512_drop_wd", {"hidden": 512, "dropout": 0.5, "weight_decay": 1e-2}),
}
for label, (name, change) in GRID.items():
    record(f"features: {label}", name, {**FEATURE_BASE, **change}, build_feature_mlp, lambda config: features)

feature_table = table([f"features: {label}" for label in GRID])
feature_table.loc["random forest (P11), 5 random states"] = [np.mean(forest_accs), np.std(forest_accs, ddof=1),
                                                              min(forest_accs), max(forest_accs), np.nan]
feature_table

===== features: linear
9015 parameters | before training: loss 1.774 (ln 5 = 1.609), accuracy 17.7%
epoch   1 | train loss 0.905 acc 0.643 | val loss 1.138 acc 0.562
epoch   5 | train loss 0.559 acc 0.824 | val loss 1.204 acc 0.542
epoch  10 | train loss 0.428 acc 0.877 | val loss 1.366 acc 0.527


epoch  15 | train loss 0.337 acc 0.920 | val loss 1.500 acc 0.531
epoch  20 | train loss 0.280 acc 0.946 | val loss 1.617 acc 0.531
epoch  25 | train loss 0.236 acc 0.959 | val loss 1.755 acc 0.533
epoch  30 | train loss 0.201 acc 0.980 | val loss 1.859 acc 0.517
epoch  35 | train loss 0.173 acc 0.990 | val loss 2.000 acc 0.523
epoch  40 | train loss 0.154 acc 0.991 | val loss 2.110 acc 0.519


epoch  45 | train loss 0.130 acc 0.995 | val loss 2.205 acc 0.519
epoch  50 | train loss 0.115 acc 0.998 | val loss 2.307 acc 0.515

best validation accuracy 56.2% at epoch 1 (0 s)
seed 1 (1 of 4 more): training, about 0 s...


   best validation accuracy 58.1% at epoch 2
seed 2 (2 of 4 more): training, about 0 s...


   best validation accuracy 57.7% at epoch 2
seed 3 (3 of 4 more): training, about 0 s...


...

   best validation accuracy 60.0% at epoch 9
seed 2 (2 of 4 more): training, about 3 s...


   best validation accuracy 60.6% at epoch 1
seed 3 (3 of 4 more): training, about 3 s...


   best validation accuracy 59.4% at epoch 3
seed 4 (4 of 4 more): training, about 3 s...


   best validation accuracy 61.7% at epoch 1
saved results/features_mlp512_drop_wd_2026-10-26_18-44-53.pkl and added a row to results/experiments.csv
===== features: 512, dropout 0.5, wd 1e-2: mean 60.3% (std 0.9%, 59.4% to 61.7%)



,val mean (5 seeds),std,min,max,trained weights
network,,,,,
features: linear,0.575000,0.008000,0.562000,0.5830,9015.0
"features: linear, wd 1e-2",0.578000,0.009000,0.562000,0.5850,9015.0
features: 256,0.602000,0.004000,0.596000,0.6060,462853.0
"features: 256, wd 1e-2",0.601000,0.002000,0.598000,0.6040,462853.0
"features: 256, dropout 0.5",0.599000,0.005000,0.592000,0.6060,462853.0
"features: 256, dropout 0.5, wd 1e-2",0.605000,0.005000,0.596000,0.6080,462853.0
"features: 512, dropout 0.5",0.601000,0.011000,0.594000,0.6170,925701.0
"features: 512, dropout 0.5, wd 1e-2",0.603000,0.009000,0.594000,0.6170,925701.0
"random forest (P11), 5 random states",0.607083,0.023781,0.577083,0.6375,NaN


In [4]:
chosen = feature_table.drop(index="random forest (P11), 5 random states")["val mean (5 seeds)"].idxmax()
chosen_stem = runs[chosen]["stem"]
print(f"chosen on validation: {chosen} (run {chosen_stem})")

row = summary().set_index("file").loc[chosen_stem]
if pd.isna(row["test_acc"]):  # museum B once (ADR 0003); a second run of this notebook only reads it
    test_loss, test_acc = score_on_test(chosen_stem)
else:
    test_loss, test_acc = row["test_loss"], row["test_acc"]
print(f"museum B: feed-forward network on the handcrafted features {test_acc:.1%} (loss {test_loss:.3f}) "
      f"| random forest on the same features (P11) 58.5%")

chosen on validation: features: 256, dropout 0.5, wd 1e-2 (run features_mlp256_drop_wd_2026-10-26_18-44-27)
features_mlp256_drop_wd_2026-10-26_18-44-27: test accuracy 62.7%, test loss 0.985 (written in results/experiments.csv)
museum B: feed-forward network on the handcrafted features 62.7% (loss 0.985) | random forest on the same features (P11) 58.5%


**What we see.**

- **The forest is noisier than P11 showed.** With the same settings and 5 random states it ranges from 57.7% to 63.7% on validation (mean 60.7%, std 2.4). Random state 0 gives 60.4% here against 60.0% recorded in P11 (2 images: the same kind of drift between computers or library versions as the dense network, ADR 0015).
- **Without a hidden layer** (a logistic regression trained by gradient descent) the network reaches 57.5%, like the logistic regression of P11 (57.7% at C = 0.001).
- **With a hidden layer it matches the forest:** the six hidden-layer networks all end between 59.9% and 60.5% (std 0.2 to 1.1), against 60.7% for the forest's mean. The grid barely matters: dropout and weight decay move the mean by less than a point.
- They learn very fast and memorise: the chosen network (256 ReLUs, dropout 0.5, weight decay 1e-2) has its best validation epoch at 3 (seed 0) and ends at 99.7% train accuracy.
- **Museum B, once: 62.7%** (loss 0.985), against **58.5%** for the forest on the same features. A gap of 4.2 points is the size of the noise (about +/- 4 points on 600 paintings for each score), and the forest's own spread across random states (std 2.4 on validation) is of the same order: museum B does not show a real difference.
- So at equal inputs, the feed-forward network is as good as the forest, not better: **the features decide, not the kind of model.** The dense network of P13 stayed at 40% because its inputs were raw pixels.

## 2. A feed-forward head on the frozen ResNet-18

The 512 numbers that the frozen ResNet-18 gives for each painting (P06, computed once), with three heads: the single linear
layer of P06 (re-run on this computer), and a hidden layer of 256 ReLUs before it, without and with dropout.

In [5]:
RESNET_BASE = {"model": "resnet18", "image_size": 224, "finetune": "head", "epochs": 30, "lr": 1e-3, "batch_size": 64,
               "weight_decay": 0.0, "dropout": 0.0, "seed": 0}
start = time.time()
resnet_features = load_resnet_data(RESNET_BASE)
print(f"frozen ResNet-18 features: train {tuple(resnet_features['X_train'].shape)} ({time.time() - start:.0f} s)")

HEADS = {"linear head (re-run)": ("resnet18_head_rerun", {}),
         "256 hidden": ("resnet18_mlp_head", {"head_hidden": 256}),
         "256 hidden, dropout 0.5": ("resnet18_mlp_head_drop", {"head_hidden": 256, "dropout": 0.5})}
for label, (name, change) in HEADS.items():
    record(f"ResNet-18: {label}", name, {**RESNET_BASE, **change}, build_resnet_head, lambda config: resnet_features)

head_table = table([f"ResNet-18: {label}" for label in HEADS])
base = head_table.loc["ResNet-18: linear head (re-run)", "val mean (5 seeds)"]
head_table["gain"] = head_table["val mean (5 seeds)"] - base
head_table["counts (> +2 points)"] = head_table["gain"] > BAR
head_table.round(3)

frozen ResNet-18 features: train (1920, 512) (43 s)
===== ResNet-18: linear head (re-run)
2565 parameters | before training: loss 1.877 (ln 5 = 1.609), accuracy 12.6%
epoch   1 | train loss 0.825 acc 0.763 | val loss 0.869 acc 0.735
epoch   5 | train loss 0.450 acc 0.844 | val loss 0.546 acc 0.787
epoch  10 | train loss 0.373 acc 0.876 | val loss 0.526 acc 0.794
epoch  15 | train loss 0.328 acc 0.889 | val loss 0.529 acc 0.792
epoch  20 | train loss 0.282 acc 0.908 | val loss 0.521 acc 0.802


epoch  25 | train loss 0.265 acc 0.917 | val loss 0.536 acc 0.783
epoch  30 | train loss 0.237 acc 0.933 | val loss 0.539 acc 0.796

best validation accuracy 80.2% at epoch 20 (0 s)
seed 1 (1 of 4 more): training, about 0 s...


   best validation accuracy 80.2% at epoch 6
seed 2 (2 of 4 more): training, about 0 s...


   best validation accuracy 81.0% at epoch 8
seed 3 (3 of 4 more): training, about 0 s...


   best validation accuracy 80.4% at epoch 8
seed 4 (4 of 4 more): training, about 0 s...


...

   best validation accuracy 81.2% at epoch 18
seed 2 (2 of 4 more): training, about 0 s...


   best validation accuracy 81.0% at epoch 6
seed 3 (3 of 4 more): training, about 0 s...


   best validation accuracy 80.6% at epoch 14
seed 4 (4 of 4 more): training, about 0 s...


   best validation accuracy 80.4% at epoch 16
saved results/resnet18_mlp_head_drop_2026-10-26_18-45-42.pkl and added a row to results/experiments.csv
===== ResNet-18: 256 hidden, dropout 0.5: mean 80.8% (std 0.3%, 80.4% to 81.2%)



,val mean (5 seeds),std,min,max,trained weights,gain,counts (> +2 points)
network,,,,,,,
ResNet-18: linear head (re-run),0.803,0.005,0.798,0.810,2565,0.000,False
ResNet-18: 256 hidden,0.803,0.005,0.796,0.808,132613,0.000,False
"ResNet-18: 256 hidden, dropout 0.5",0.808,0.003,0.804,0.812,132613,0.005,False


In [6]:
counting = head_table.index[head_table["counts (> +2 points)"]].tolist()
print("feed-forward heads that count:", counting if counting else "none: the linear head stays, museum B is not used")
for label in counting:  # only a head that counts is scored on museum B
    print(label, score_on_test(runs[label]["stem"]))

feed-forward heads that count: none: the linear head stays, museum B is not used


**What we see.**

- **The re-run linear head gives exactly the recorded 80.3%** (5 seeds, std 0.5).
- **A hidden layer adds nothing:** 80.3% without dropout, 80.8% with dropout 0.5 (+0.5 points), far below the 2-point bar, for 52 times the trained weights (132,613 against 2,565). No head counts: the linear head stays, and museum B was not used.
- This agrees with P08 (ADR 0011): several layers, augmentation and a ResNet-50 also stayed within 2 points. On these 512 numbers the five genres are already almost separated by straight lines; what is left (P10) looks like overlapping labels, which no head can fix.

## Saved here

Eleven rows in `results/experiments.csv` (names `features_*` and `resnet18_*_rerun` / `resnet18_mlp_head*` of this date, 5 seeds
each), and the museum B score of the chosen network on the handcrafted features only.

## What we see

| inputs | feed-forward network | validation (5 seeds) | museum B | same inputs, other model |
|---|---|---|---|---|
| raw pixels, 32 px (P02, P13) | 1 hidden layer, mirrored copies | 42.5% | 41.0% | |
| handcrafted colour + edges (P11) | 1 hidden layer of 256, dropout 0.5, wd 1e-2 | 60.5% | **62.7%** | random forest: 60.7% (5 states), 58.5% on museum B |
| frozen ResNet-18 (P06) | 1 hidden layer of 256 + dropout 0.5 | 80.8% | not scored | linear head: 80.3%, 84.5% on museum B |

- **The same kind of network goes from 41% to 63% to 80% depending only on what it is given.** On raw pixels it cannot generalise (P13); on handcrafted features it equals a random forest; on pretrained features a single linear layer is already enough.
- **For the director:** "neural network or classical model?" is the wrong question at equal inputs: the feed-forward network and the forest tie. What pays is the features, and the best features here are the ones a network learnt on ImageNet (ADR 0014).

**Limits.** One feature set each; the forest is the P11 one, not tuned; the feed-forward network on handcrafted features was chosen among eight networks on validation and scored once on museum B, as was the forest; museum B differences of a few points are inside the noise. A scratch run of the same grids, unsaved, preceded this notebook.